# Exploratory analysis of customer behavior

This notebook assesses the prepared customer-level features before clustering. All transformations are diagnostic and leave `customer_features.csv` unchanged.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
data_path = project_root / 'data' / 'processed' / 'customer_features.csv'
customers = pd.read_csv(data_path)
features = [
    'Recency', 'Frequency', 'MonetaryValue', 'TotalItems',
    'UniqueProducts', 'AverageOrderValue', 'AverageItemsPerOrder',
    'CustomerLifetimeDays',
]

## 1. Dataset overview

In [ ]:
overview = pd.Series({
    'Customers': len(customers),
    'Columns': customers.shape[1],
    'Missing values': int(customers.isna().sum().sum()),
    'Duplicate CustomerID values': int(customers['CustomerID'].duplicated().sum()),
})
display(overview.to_frame('Value'))
display(customers.head())

## 2. Distribution analysis

In [ ]:
distribution_summary = customers[features].describe().T
distribution_summary['skewness'] = customers[features].skew()
display(distribution_summary)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for feature, axis in zip(features, axes.flat):
    axis.hist(customers[feature], bins=40, color='#2563eb', edgecolor='white')
    axis.set_title(feature)
    axis.set_xlabel('Observed value')
    axis.set_ylabel('Customers')
fig.suptitle('Customer feature distributions', fontsize=14)
fig.tight_layout()
plt.show()

## 3. Outlier analysis

In [ ]:
outlier_rows = []
for feature in features:
    q1 = customers[feature].quantile(0.25)
    q3 = customers[feature].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outside = (customers[feature] < lower) | (customers[feature] > upper)
    outlier_rows.append({
        'Feature': feature, 'Q1': q1, 'Q3': q3, 'IQR': iqr,
        'LowerBound': lower, 'UpperBound': upper,
        'OutsideBounds': int(outside.sum()),
        'PercentOutside': outside.mean() * 100,
    })
outlier_summary = pd.DataFrame(outlier_rows).set_index('Feature')
display(outlier_summary.round(3))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for feature, axis in zip(features, axes.flat):
    axis.boxplot(customers[feature], vert=False, showfliers=True)
    axis.set_title(feature)
fig.suptitle('IQR-based view of extreme observations', fontsize=14)
fig.tight_layout()
plt.show()

IQR flags are descriptive, not deletion rules. The retailer serves wholesale customers, so extreme positive values may represent genuine high-volume behavior.

## 4. Correlation analysis

In [ ]:
correlations = customers[features].corr(method='pearson')
display(correlations.round(3))
fig, axis = plt.subplots(figsize=(10, 8))
image = axis.imshow(correlations, cmap='RdBu_r', vmin=-1, vmax=1)
axis.set_xticks(range(len(features)), features, rotation=45, ha='right')
axis.set_yticks(range(len(features)), features)
for row in range(len(features)):
    for col in range(len(features)):
        axis.text(col, row, f'{correlations.iloc[row, col]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(image, ax=axis, label='Pearson correlation')
axis.set_title('Correlation between behavioral features')
fig.tight_layout()
plt.show()

## 5. Log-transformation comparison

In [ ]:
log_skewness = pd.DataFrame({
    'Before': customers[features].skew(),
    'After log1p': np.log1p(customers[features]).skew(),
})
display(log_skewness.round(3))
log_skewness.plot.bar(figsize=(12, 5), color=['#64748b', '#2563eb'])
plt.axhline(0, color='black', linewidth=0.8)
plt.ylabel('Skewness')
plt.title('Skewness before and after log1p')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 6. Country analysis

In [ ]:
country_counts = customers['Country'].value_counts()
uk_share = country_counts.get('United Kingdom', 0) / len(customers) * 100
print(f'Countries: {country_counts.size}')
print(f'United Kingdom share: {uk_share:.2f}%')
display(country_counts.head(10).to_frame('Customers'))
country_counts.head(10).sort_values().plot.barh(figsize=(9, 5), color='#2563eb')
plt.xlabel('Customers')
plt.title('Top 10 countries by customer count')
plt.tight_layout()
plt.show()

Country is not encoded here. Numeric labels would invent an ordering, while one-hot encoding could add many sparse dimensions and emphasize the large UK majority.

## 7. Customer behavior checks

In [ ]:
print('Customers with Frequency = 1:', int((customers['Frequency'] == 1).sum()))
print('Customers with CustomerLifetimeDays = 0:', int((customers['CustomerLifetimeDays'] == 0).sum()))
display(customers.nlargest(5, 'MonetaryValue')[['CustomerID', 'Country', 'Frequency', 'MonetaryValue', 'TotalItems']])
display(customers.nlargest(5, 'TotalItems')[['CustomerID', 'Country', 'Frequency', 'MonetaryValue', 'TotalItems']])

## 8. Preliminary observations

- Six purchasing-volume variables are strongly right-skewed and benefit from diagnostic `log1p` transformation.
- MonetaryValue and TotalItems are strongly correlated, so using both may overweight overall customer size.
- Recency and CustomerLifetimeDays do not show evidence requiring automatic log transformation.
- A candidate behavioral set is Recency, Frequency, MonetaryValue, UniqueProducts, and CustomerLifetimeDays. This is a recommendation for later evaluation, not a final clustering decision.
- StandardScaler alone does not correct skew. Log-transforming strongly skewed selected variables before scaling is the more defensible starting point.